In [1]:
import os

os.chdir("..")

In [2]:
import tensorflow as tf

In [3]:
model= tf.keras.models.load_model("artifacts/training/model.h5")

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class EvaluationConfig:
    root_dir: Path
    model_path: Path
    training_data: Path
    all_params: dict
    params_image_size: list
    params_batch_size: int


In [4]:
from cnnClassifer.constants.paths import *
from cnnClassifer.utils.common import read_yaml,create_directories,save_json

In [5]:
class ConfigurationManager:
    def __init__(self,config_filepath=CONFIG_FILE_PATH,params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_evaluation_config(self) -> EvaluationConfig:
        
        root_=self.config.evaluation

        evaluation_config = EvaluationConfig(
            root_dir= root_.root_dir,
            model_path = "artifacts/training/model.h5",
            training_data= "artifacts/data_ingestion/Human-Face",
            all_params=self.params,
            params_batch_size=self.params.BATCH_SIZE,
            params_image_size=self.params.IMAGE_SIZE
        )

        return evaluation_config
    

        

In [6]:
class Evaluation:

    def __init__(self, config: EvaluationConfig):
        self.config = config

        create_directories([self.config.root_dir])

    def _valid_generator(self):

        datagenerator_kwargs = dict(
            rescale=1./255,
            validation_split=0.3
        )

        dataflow_kwargs = dict(
            target_size=tuple(self.config.params_image_size[:2]),
            batch_size=self.config.params_batch_size,
            interpolation="bilinear"
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

    @staticmethod
    def load_model(path: Path):
        return tf.keras.models.load_model(path)

    def evaluation(self):
        self.model = self.load_model(self.config.model_path)

        self._valid_generator()

        self.score = self.model.evaluate(
            self.valid_generator
        )

    def save_score(self):
        scores = {
            "loss": float(self.score[0]),
            "accuracy": float(self.score[1])
        }

        save_json(
            path=Path("artifacts/evaluation/score.json"), data=scores)

In [7]:
try:
    config = ConfigurationManager()
    val_config = config.get_evaluation_config()
    evaluation = Evaluation(val_config)
    evaluation.evaluation()
    evaluation.save_score()

except Exception as e:
    raise e

[2026-06-21 14:33:47,578: INFO: common: YAML file: config\config.yaml loaded successfully]
[2026-06-21 14:33:47,583: INFO: common: YAML file: params.yaml loaded successfully]
[2026-06-21 14:33:47,584: INFO: common: Directory created at: artifacts]
[2026-06-21 14:33:47,586: INFO: common: Directory created at: artifacts/evaluation]
[2026-06-21 14:33:48,038: WARNING: config: TensorFlow GPU support is not available on native Windows for TensorFlow >= 2.11. Even if CUDA/cuDNN are installed, GPU will not be used. Please use WSL2 or the TensorFlow-DirectML plugin.]
[2026-06-21 14:33:48,054: WARNING: saving_utils: Compiled the loaded model, but the compiled metrics have yet to be built. `model.compile_metrics` will be empty until you train or evaluate the model.]
Found 152 images belonging to 2 classes.
10/10 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 1.0000 - loss: 0.0145
[2026-06-21 14:34:02,901: INFO: common: JSON file saved at: artifacts\evaluation\score.json]
